# Model Evaluation & Analysis

Comprehensive evaluation and analysis of trained emotion recognition models on the test set.

**Contents:**
- Load all trained models
- Evaluate each model on test set
- Generate detailed evaluation metrics
- Create visualization plots
- Compare model performance
- Analyze failure cases

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import load_model

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_per_class_metrics, plot_prediction_distribution

sns.set_style("whitegrid")

EMOTION_LABELS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

## Load Data and Models

In [ ]:
# Load test data
data_dir = os.path.join('..', 'data', 'preprocessed')
X_test = np.load(os.path.join(data_dir, 'X_test.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test.npy'))

print(f"Test data loaded: {X_test.shape}")

# Load trained models
models_dir = os.path.join('..', 'saved_models')
available_models = [f for f in os.listdir(models_dir) if f.endswith('_best.keras')]
print(f"\nAvailable models: {available_models}")

## Evaluate All Models

In [ ]:
# Dictionary to store results
results = {}

print("Evaluating all trained models...")
print("=" * 60)

for model_file in available_models:
    model_name = model_file.replace('_best.keras', '')
    model_path = os.path.join(models_dir, model_file)
    
    print(f"\n{model_name.upper()}")
    print("-" * 60)
    
    # Load and evaluate
    model = load_model(model_path)
    results[model_name] = evaluate_model(model, X_test, y_test, model_name=model_name)
    
    # Generate visualizations
    plot_confusion_matrix(y_test, results[model_name]['y_pred'], model_name=model_name)
    plot_per_class_metrics(y_test, results[model_name]['y_pred'], model_name=model_name)
    plot_prediction_distribution(results[model_name]['y_pred_proba'], y_test, model_name=model_name)

## Comparison Summary

In [ ]:
# Create comparison DataFrame
comparison_data = []
for model_name, result in results.items():
    comparison_data.append({
        'Model': model_name,
        'Test Accuracy': result['accuracy']
    })

comparison_df = pd.DataFrame(comparison_data).sort_values('Test Accuracy', ascending=False)

print("\n" + "=" * 60)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 60)
print(comparison_df.to_string(index=False))

# Identify best model
best_model_name = comparison_df.iloc[0]['Model']
best_accuracy = comparison_df.iloc[0]['Test Accuracy']
print(f"\n🏆 Best Model: {best_model_name} (Accuracy: {best_accuracy:.4f})")

## Visualization: Model Comparison

In [ ]:
# Plot model comparison
fig, ax = plt.subplots(figsize=(10, 6))

models_list = comparison_df['Model'].tolist()
accuracies = comparison_df['Test Accuracy'].tolist()
colors = ['#2ecc71' if acc == max(accuracies) else '#3498db' for acc in accuracies]

bars = ax.barh(models_list, accuracies, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    ax.text(acc + 0.005, bar.get_y() + bar.get_height()/2, 
            f'{acc:.4f}', va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('Test Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xlim([min(accuracies) * 0.95, max(accuracies) * 1.05])
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/model_accuracy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Model comparison plot saved")

## Analysis by Emotion Class

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Get best model results
best_model_result = results[best_model_name]
y_pred = best_model_result['y_pred']

# Calculate metrics per emotion
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred)

# Create per-emotion analysis
emotion_metrics = pd.DataFrame({
    'Emotion': EMOTION_LABELS,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

print("\nPER-EMOTION METRICS (Best Model):")
print("=" * 80)
print(emotion_metrics.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(EMOTION_LABELS))
width = 0.25

ax.bar(x - width, precision, width, label='Precision', alpha=0.8)
ax.bar(x, recall, width, label='Recall', alpha=0.8)
ax.bar(x + width, f1, width, label='F1-Score', alpha=0.8)

ax.set_xlabel('Emotion', fontweight='bold')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title('Per-Emotion Performance Metrics', fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(EMOTION_LABELS, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/per_emotion_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary Report

In [ ]:
# Generate comprehensive summary report
summary_report = f"""
{'='*80}
FACIAL EMOTION RECOGNITION - EVALUATION SUMMARY
{'='*80}

PROJECT OVERVIEW:
- Task: Multi-class emotion classification (7 emotions)
- Dataset: FER2013 from Kaggle
- Test Set Size: {len(y_test)} samples

MODELS EVALUATED:
- Total models trained: {len(results)}
- Models compared: {', '.join(results.keys())}

BEST MODEL PERFORMANCE:
- Model: {best_model_name}
- Test Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)

EMOTION CLASS DISTRIBUTION (Test Set):
"""

for emotion_idx, label in enumerate(EMOTION_LABELS):
    count = (y_test == emotion_idx).sum()
    pct = count / len(y_test) * 100
    summary_report += f"\n  - {label:10s}: {count:5d} samples ({pct:5.2f}%)"

summary_report += f"""

BEST MODEL METRICS:
- Macro-Average Precision: {precision.mean():.4f}
- Macro-Average Recall: {recall.mean():.4f}
- Macro-Average F1-Score: {f1.mean():.4f}

NEXT STEPS:
1. Deploy best model: {best_model_name}
2. Test on real-world webcam input using demo/webcam_demo.py
3. Consider fine-tuning on domain-specific data for better generalization
4. Explore ensemble methods combining multiple models for improved robustness

{'='*80}
"""

print(summary_report)

# Save report
with open('../results/evaluation_report.txt', 'w') as f:
    f.write(summary_report)

print("✓ Report saved to results/evaluation_report.txt")